# Multi Agent Systems & Workflow Patterns

In [48]:
import os

## Verify and Authenticate the ADK

In [49]:
def verify_and_authenticate():
    api_key = os.environ.get("GOOGLE_API_KEY")
    if api_key:
        print("Key is configured")
        return
    print("Key not found")
    return 

In [50]:
verify_and_authenticate()

Key is configured


## Importing ADK Components

In [51]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types

print("ADK Components are imported successfully")

ADK Components are imported successfully


## Configure retry options

In [52]:
retry_config = types.HttpRetryOptions(attempts=5,expBase=7,initialDelay=1,httpStatusCodes=[429,500,503,504])

## Example 1: Reasearch & Summarization Agent

Let's build a system with two specialized agents:

- *Research Agent* - Searches for information using Google Search
- *Summarizer Agent* - Creates concise summaries from research findings

### Reasearch Agent

In [53]:
research_agent = Agent(
    name = "ResearchAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction=""" You are a specialized research agent. Your only job is to use the 
    google_search tool to find 2-3 pieces of relevant information on the given topic and present
    the findings with citations""",
    output_key="research_findings"
)
print("Research Agent Created")

Research Agent Created


### Summarizer Agent

In [54]:
summarizer_agent = Agent(
    name="SummarizerAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction=""" Read the provided research findings: {research_findings}
    Create a concise summary as a bulleted list with 3-5 key points
    """,
    output_key="final_summary"
)
print("summarizer_agent created")

summarizer_agent created


### Research Coordinator Agent

Orchestrates the workflow by calling the sub-agents as tools

In [55]:
root_agent = Agent(
    name="ResearchCoordinator",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction=""" You are a research coordinator. Your goal is to answer the user's query by orchestrating a workflow.
    1. First, you MUST call the `ResearchAgent` tool to find the relavent information on the topic provided by the user.
    2. Next, after receiving the research findings, you MUST call the `SummarizerAgent` tool to create a concise summary.
    3. Finally, present the final summary clearly to the user as your response.""",
    tools=[AgentTool(research_agent), AgentTool(summarizer_agent)]
)

print("root_agent created")

root_agent created


### Configuring the runner

In [56]:
runner = InMemoryRunner(agent=root_agent)

In [57]:
await runner.run_debug(
    "What are the latest advancements in quantum computing and what do they mean for AI?"
)


 ### Created new session: debug_session_id

User > What are the latest advancements in quantum computing and what do they mean for AI?


[Event(model_version='gemini-2.5-flash-lite', content=Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'request': 'Latest advancements in quantum computing and their implications for AI'
         },
         id='adk-3b2aa820-ae05-4dcd-b70a-2eacfa466aa4',
         name='ResearchAgent'
       )
     ),
   ],
   role='model'
 ), grounding_metadata=None, partial=None, turn_complete=None, finish_reason=<FinishReason.STOP: 'STOP'>, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=GenerateContentResponseUsageMetadata(
   candidates_token_count=23,
   prompt_token_count=192,
   prompt_tokens_details=[
     ModalityTokenCount(
       modality=<MediaModality.TEXT: 'TEXT'>,
       token_count=192
     ),
   ],
   total_token_count=215
 ), live_session_resumption_update=None, input_transcription=None, output_transcription=None, avg_logprobs=None, logprobs_result=None, cache_metadata=None, citation_metadata=None,

## Sequential Workflows - The Assembly Line

### Example Blog Post Creation with Sequential Agents

Let's build a system with three specialized agents:
1. **Outline Agent** - Creates a blog outline for a given topic.
2. **Writer Agent** - Writes a blog post
3. **Editor Agent** - Edits a blog post draft for clarity and structure

#### Outline Agent: Creates the initial blog post outline

In [58]:
outline_agent = Agent(
    name="OutlineAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction=""" Create a blog outline for the given topic with:
    1. A catchy headline.
    2. An Intruduction hook
    3. 3-5 main sections with 2-3 bullet points for each
    4. A concluding thought """,
    output_key="blog_outline"
)
print("outline_agent created")

outline_agent created


#### Writer Agent: Edits and polishes the draft from the writer agent

In [59]:
writer_agent = Agent(
    name="WriterAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction=""" Following this outline strictly: {blog_outline}
    write a brief, 200 to 300-word blog post with an engaging and informative tone.""",
    output_key="blog_draft"
)

print("writer_agent created")

writer_agent created


#### Editor Agent: Edits and ploshes the fraft from the writer agent

In [60]:
editor_agent = Agent(
    name="EditorAgent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""Edit this draft: {blog_draft}
    Your task is to polish the test by fixing any grammatical errors, improving the flow and sentence structure, and enhancing overall clarity.""",
    output_key="final_blog"
)
print("editor agent created")

editor agent created


#### Bringing Agents together By using the sequential agent tool

In [61]:
seq_agent = SequentialAgent(
    name="BlogPipeline",
    sub_agents=[outline_agent, writer_agent, editor_agent]
)
print("Pipeline created")

Pipeline created


#### Configuring the runner for blog post creator

In [62]:
seq_runner = InMemoryRunner(agent=seq_agent)

In [63]:
await seq_runner.run_debug(
    """Write a blog post about the benifits of multi-agent systems for the software developers, include outline and basic syntax to create a multi-agent systems"""
)


 ### Created new session: debug_session_id

User > Write a blog post about the benifits of multi-agent systems for the software developers, include outline and basic syntax to create a multi-agent systems
OutlineAgent > ## Outline:

**Headline:** Beyond the Single Thread: Unleash Superpowers with Multi-Agent Systems (MAS)

**Introduction Hook:**
Tired of wrestling with monolithic codebases and complex, single-threaded logic? Imagine a world where your software isn't just a program, but a bustling ecosystem of intelligent, collaborating "agents" working seamlessly to solve problems. Welcome to the exciting realm of Multi-Agent Systems (MAS), and discover how they can revolutionize your development workflow.

---

**Main Sections:**

**1. The Power of Decentralization: Smarter, More Resilient Software**

*   **Enhanced Modularity and Reusability:** Break down complex tasks into smaller, independent agents, making your code easier to understand, debug, and reuse across different projects

[Event(model_version='gemini-2.5-flash-lite', content=Content(
   parts=[
     Part(
       text="""## Outline:
 
 **Headline:** Beyond the Single Thread: Unleash Superpowers with Multi-Agent Systems (MAS)
 
 **Introduction Hook:**
 Tired of wrestling with monolithic codebases and complex, single-threaded logic? Imagine a world where your software isn't just a program, but a bustling ecosystem of intelligent, collaborating "agents" working seamlessly to solve problems. Welcome to the exciting realm of Multi-Agent Systems (MAS), and discover how they can revolutionize your development workflow.
 
 ---
 
 **Main Sections:**
 
 **1. The Power of Decentralization: Smarter, More Resilient Software**
 
 *   **Enhanced Modularity and Reusability:** Break down complex tasks into smaller, independent agents, making your code easier to understand, debug, and reuse across different projects.
 *   **Improved Robustness and Fault Tolerance:** If one agent fails, the system can often adapt and conti

## Parallel Workflows - Independent Researchers

**The Problems and Bottleneck over sequential workflows**

The previous sequential agent is great, but it's an assembly line. Each step must wait for the previous one to finish. What if you have several tasks that are not dependent on each other? For example, researching three different topics. Running them in sequence would be slow and inefficient, creating a bottleneck where each task waits unnecessarily.

**The solution: Concurrent Executions**

When you have independent tasks, you can run them all at the same time using a ParallelAgent. This agent executes all of its sub-agents concurrently, dramatically speeding up the workflow. Once all parallel tasks are complete, you can then pass their combined results to a final 'aggregator' step.

**Use Parallel when**: Tasks are Independent, speed matters and you can execute concurrently.

### Example: Parallel Multi Topic Researcher

Let's build a system with four agents:
1. **Tech Researcher**: Researches AI/ML news and Trends.
2. **Health Researcher**: Researches recent medical news and trends.
3. **Finance Researcher**: Researches finance and fintech news and trends.
4. **Aggregator Agent**: Combines all research findings into a single summary.

#### Tech researcher: Focuses on AI and ML trends

In [64]:
tech_researcher = Agent(
    name="TechResearcher",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""
    Research the latest AI/ML trends. Include 3 key developments,
    the main companies involved, and the potential impact. Keep the report cery concise (100 words).
    """,
    tools=[google_search],
    output_key="tech_researcher"
)

print("tech_research agent created")

tech_research agent created


#### Health Researcher: Focuses on medical breakthroughs

In [65]:
health_researcher = Agent(
    name="HealthResearcher",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""
    Research recent medical breakthroughs. Include 3 significant advances,
    their practical applications, and estimated timelines. Keep the report concise (100 words).
    """,
    tools=[google_search],
    output_key="health_researcher"
)
print("health_researcher created")

health_researcher created


#### Finance Researcher: Focuses on the fintech trends

In [66]:
finance_researcher = Agent(
    name="FinanceResearcher",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""
    Research current fintech trends. Include 3 key trends,
    their market applications and the future outlook. Keep the report concise (100 words).
    """,
    output_key="finance_researcher"
)
print("finance_researcher created")

finance_researcher created


#### The Aggregator Agent runs *after* the parallel step t0 synthesize the results.

In [67]:
aggregator_agent = Agent(
    name="AggregatorAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""
    Combine these three research findings into a single executive summary:

    ** Technology Trends **
    {tech_researcher}

    ** Health Trends **
    {health_researcher}

    ** Finance Trends **
    {finance_researcher}

    Your summary should highlight common themes, surprising connections, and the important key takeaways from all three reports.
    The final summary should be around 200 words.
    """,
    output_key="executive_summary"
)
print("aggregator_agent created")

aggregator_agent created


#### The parallel agent runs all the sub-agents simultaneously

In [68]:
parallel_research_team = ParallelAgent(
    name="ParallelResearchTeam",
    sub_agents=[tech_researcher, health_researcher,finance_researcher]
)

This sequential agent defines the high-level workflow: run the parallel team first, then run the aggregator

In [69]:
research_team = SequentialAgent(
    name="ResearchSystem",
    sub_agents=[parallel_research_team, aggregator_agent]
)

#### Configuring the runner

In [70]:
run_research = InMemoryRunner(agent=research_team)

In [71]:
await run_research.run_debug("Run the daily executive breifing on Tech, Health and Finance")


 ### Created new session: debug_session_id

User > Run the daily executive breifing on Tech, Health and Finance
FinanceResearcher > Here's your executive briefing for today:

**Technology:** **AI Integration Deepens:** Generative AI is moving beyond chatbots to assist in software development, content creation, and complex data analysis. Market applications include accelerated code writing, personalized marketing campaigns, and advanced scientific research. Outlook: Expect wider adoption across industries, leading to increased efficiency and innovation, but also raising concerns about job displacement and ethical AI use.

**Health:** **Personalized Medicine Gains Traction:** Advancements in genomics and data analytics are enabling treatments tailored to individual patient profiles. Market applications span targeted cancer therapies, predictive disease risk assessments, and optimized drug development. Outlook: This trend promises more effective and less invasive healthcare, but faces ch

[Event(model_version='gemini-2.5-flash-lite', content=Content(
   parts=[
     Part(
       text="""Here's your executive briefing for today:
 
 **Technology:** **AI Integration Deepens:** Generative AI is moving beyond chatbots to assist in software development, content creation, and complex data analysis. Market applications include accelerated code writing, personalized marketing campaigns, and advanced scientific research. Outlook: Expect wider adoption across industries, leading to increased efficiency and innovation, but also raising concerns about job displacement and ethical AI use.
 
 **Health:** **Personalized Medicine Gains Traction:** Advancements in genomics and data analytics are enabling treatments tailored to individual patient profiles. Market applications span targeted cancer therapies, predictive disease risk assessments, and optimized drug development. Outlook: This trend promises more effective and less invasive healthcare, but faces challenges in data privacy, cos

## Loop Workflows - The Refinement Cycle

**The Problem: One-Shot Quality**

All the workflows we've seen so far run from start to finish. The SequentialAgent and ParallelAgent produce their final output and then stop. This 'one-shot' approach isn't good for tasks that require refinement and quality control. What if the first draft of our story is bad? We have no way to review it and ask for a rewrite.

**The Solution: Iterative Refinement**

When a task needs to be improved through cycles of feedback and revision, you can use a LoopAgent. A LoopAgent runs a set of sub-agents repeatedly until a specific condition is met or a maximum number of iterations is reached. This creates a refinement cycle, allowing the agent system to improve its own work over and over.

**Use Loop when:** Iterative improvement is needed, quality refinement matters, or you need repeated cycles.

### Example Story Refienment

#### Writer Agent - Writes a draft of a short story

In [72]:
initial_writer_agent = Agent(
    name="InitialWriterAgent",
    model=Gemini(
        model='gemini-2.5-flash-lite',
        retry_options=retry_config
    ),
    instruction=""" Based on the user's prompt, write the first draft of a short story ( around 100-150 words ).
    Output only the story text, with no introduction or explanation.""",
    output_key="current_story"
)

print("initial_writer_agent created.")

initial_writer_agent created.


#### Critic Agent - Reviews and critiques the short story to suggest improvements

In [73]:
critic_agent = Agent(
    name="CriticAgent",
    model=Gemini(
        model='gemini-2.5-flash-lite',
        retry_options=retry_config
    ),
    instruction=""" You are a constructive story critic. Review the story provided below.
    Story: {current_story}

    Evaluate the story's plot characters, and pacing
    - If the story is well-written and complete, you MUST respond with the exact phrase: "APPROVED"
    - Otherwise, provide 2-3 specific, actionable suggestions from improvement.""",
    output_key='critique'
)

print("critic_agent created")

critic_agent created


#### Refiner Agent - This will be called to exit the loop

In [74]:
def exit_loop():
    """Call this function ONLY when the critique is 'APPROVED', indicating the story is finished and no more changes are needed."""
    return {"status": "approved", "message": "Story approved. Exiting refinement loop."}


print("exit_loop function created.")

exit_loop function created.


In [75]:
refiner_agent = Agent(
    name="RefinerAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""
    You are a story refiner. You have a story draft and critique.

    Story Draft: {current_story}
    Critique: {critique}

    Your task is to analyse the critique.
    - IF the critique is EXACTLY "APPROVED", you MUST call the `exit_loop` function and nothing else.
    - OTHERWISE rewrite the story draft to fully incorporate the feedback from the critique.""",
    output_key="current_story",
    tools=[FunctionTool(exit_loop)]
)
print("refiner_agent created.")

refiner_agent created.


#### Story refinement Loop - The Loop Agent contains the agents will run repeatedly: Critic -> Refiner

In [76]:
story_refienment_loop = LoopAgent(
    name="StoryRefinementLoop",
    sub_agents=[critic_agent, refiner_agent],
    max_iterations=5
)

#### Story teller agent - The Sequential root agent that defines the overall workflow: Initial Write -> Refinement Loop

In [77]:
story_teller_agent = SequentialAgent(
    name="StoryPipeline",
    sub_agents=[initial_writer_agent, story_refienment_loop],
)

print("Story teller sequential root agent created")

Story teller sequential root agent created


#### Runner creation 

In [79]:
director = InMemoryRunner(agent=story_teller_agent)

In [80]:
await director.run_debug("Write a short story about a lighthouse keeper who discovers a mysterious, glowing map")


 ### Created new session: debug_session_id

User > Write a short story about a lighthouse keeper who discovers a mysterious, glowing map
InitialWriterAgent > The salty spray kissed Elias’s weathered face as he polished the great lens. Another night, another solitary vigil. But tonight was different. A storm had washed something ashore, something that pulsed with an ethereal, blue light. It was a map, rolled tight and etched onto a material like cured kelp. As he unfurled it, strange constellations and an unknown coastline shimmered into view. A single, pulsating X marked a spot not found on any charted sea. Elias’s heart hammered against his ribs. He’d spent fifty years watching the horizon, and now, the horizon had sent him a secret.
CriticAgent > The story is well-written and complete, establishing a strong sense of atmosphere and introducing a compelling mystery. The plot is set up effectively with the discovery of the map, and Elias is a clearly defined character through his long 

[Event(model_version='gemini-2.5-flash-lite', content=Content(
   parts=[
     Part(
       text='The salty spray kissed Elias’s weathered face as he polished the great lens. Another night, another solitary vigil. But tonight was different. A storm had washed something ashore, something that pulsed with an ethereal, blue light. It was a map, rolled tight and etched onto a material like cured kelp. As he unfurled it, strange constellations and an unknown coastline shimmered into view. A single, pulsating X marked a spot not found on any charted sea. Elias’s heart hammered against his ribs. He’d spent fifty years watching the horizon, and now, the horizon had sent him a secret.'
     ),
   ],
   role='model'
 ), grounding_metadata=None, partial=None, turn_complete=None, finish_reason=<FinishReason.STOP: 'STOP'>, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=GenerateContentResponseUsageMetadata(
   candidates_token_count=125,
   prompt_token_c